In [1]:
import os, requests, time
from pathlib import Path

# Create data folder inside ml/
DATA_DIR = Path('nhanes_data')
DATA_DIR.mkdir(exist_ok=True)
print(f'Data folder: {DATA_DIR.resolve()}')

# NHANES cycles we will use
CYCLES = {
    '2011-12': 'G',
    '2013-14': 'H',
    '2015-16': 'I',
    '2017-18': 'J',
}

# Files to download per cycle
# Format: (filename_prefix, description)
FILES_PER_CYCLE = [
    ('DR1TOT', 'Dietary recall day 1 totals'),
    ('DR2TOT', 'Dietary recall day 2 totals'),
    ('DEMO',   'Demographics'),
    ('GHB',    'Glycohemoglobin (HbA1c)'),
    ('TCHOL',  'Total cholesterol'),
    ('HDL',    'HDL cholesterol'),
    ('TRIGLY', 'Triglycerides'),
    ('BMX',    'Body measures (BMI, waist)'),
    ('BPX',    'Blood pressure'),
]

BASE_URL = 'https://wwwn.cdc.gov/Nchs/Nhanes'
print('Setup complete.')


Data folder: C:\Users\obbha\OneDrive\South-asian-diet-risk.cd\ml\nhanes_data
Setup complete.


In [2]:
def download_nhanes_file(cycle_label, suffix, prefix, description):
    filename = f'{prefix}_{suffix}.XPT'
    url = f'{BASE_URL}/{cycle_label}/{filename}'
    out_path = DATA_DIR / filename

    if out_path.exists():
        print(f'  Already downloaded: {filename}')
        return True

    print(f'  Downloading {filename} ({description})...')
    try:
        r = requests.get(url, timeout=60)
        if r.status_code == 200:
            out_path.write_bytes(r.content)
            print(f'    Saved {len(r.content)//1024} KB')
            time.sleep(0.5)  # be polite to the CDC server
            return True
        else:
            print(f'    HTTP {r.status_code} — skipping')
            return False
    except Exception as e:
        print(f'    Error: {e}')
        return False

print('Download function ready.')


Download function ready.


In [3]:
success = 0
failed = []

for cycle_label, suffix in CYCLES.items():
    print(f'\nCycle {cycle_label}:')
    for prefix, description in FILES_PER_CYCLE:
        ok = download_nhanes_file(cycle_label, suffix, prefix, description)
        if ok: success += 1
        else: failed.append(f'{prefix}_{suffix}')

print(f'\nDownload complete: {success} files downloaded')
if failed:
    print(f'Failed: {failed}')
    print('Failed files are usually from cycles where that survey',
          'was not conducted. This is normal.')
else:
    print('All files downloaded successfully.')

# List what we have
files = list(DATA_DIR.glob('*.XPT'))
print(f'\nFiles in nhanes_data/: {len(files)}')
for f in sorted(files): print(f'  {f.name}')



Cycle 2011-12:
    Saved 20 KB
    Saved 20 KB
    Saved 20 KB
    Saved 20 KB
    Saved 20 KB
    Saved 20 KB
    Saved 20 KB
    Saved 20 KB
    Saved 20 KB

Cycle 2013-14:
    Saved 20 KB
    Saved 20 KB
    Saved 20 KB
    Saved 20 KB
    Saved 20 KB
    Saved 20 KB
    Saved 20 KB
    Saved 20 KB
    Saved 20 KB

Cycle 2015-16:
    Saved 20 KB
    Saved 20 KB
    Saved 20 KB
    Saved 20 KB
    Saved 20 KB
    Saved 20 KB
    Saved 20 KB
    Saved 20 KB
    Saved 20 KB

Cycle 2017-18:
    Saved 20 KB
    Saved 20 KB
    Saved 20 KB
    Saved 20 KB
    Saved 20 KB
    Saved 20 KB
    Saved 20 KB
    Saved 20 KB
    Saved 20 KB

Download complete: 36 files downloaded
All files downloaded successfully.

Files in nhanes_data/: 36
  BMX_G.XPT
  BMX_H.XPT
  BMX_I.XPT
  BMX_J.XPT
  BPX_G.XPT
  BPX_H.XPT
  BPX_I.XPT
  BPX_J.XPT
  DEMO_G.XPT
  DEMO_H.XPT
  DEMO_I.XPT
  DEMO_J.XPT
  DR1TOT_G.XPT
  DR1TOT_H.XPT
  DR1TOT_I.XPT
  DR1TOT_J.XPT
  DR2TOT_G.XPT
  DR2TOT_H.XPT
  DR2TOT_I.XPT
  DR2

In [6]:
import pyreadstat

# Test-read one file to confirm the format works
test_file = DATA_DIR / 'DR1TOT_G.XPT'
if test_file.exists():
    df, meta = pyreadstat.read_xport(str(test_file))
    print(f'DR1TOT_G.XPT loaded: {df.shape[0]} rows, {df.shape[1]} columns')
    print('Column sample:', list(df.columns[:10]))
else:
    print('DR1TOT_G.XPT not found — check download step')


ReadstatError: Invalid file, or file has unsupported features

In [7]:
import pandas as pd

# pandas can read XPT files directly without pyreadstat
test_file = DATA_DIR / 'DR1TOT_G.XPT'
df = pd.read_sas(str(test_file), format='xport', encoding='utf-8')
print(f'DR1TOT_G.XPT loaded: {df.shape[0]} rows, {df.shape[1]} columns')
print('Column sample:', list(df.columns[:10]))

ValueError: Header record is not an XPORT file.

In [8]:
import requests
from pathlib import Path

DATA_DIR = Path('nhanes_data')

def download_nhanes_v2(url, out_path):
    """Download with proper headers to avoid CDN issues."""
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36',
        'Accept': '*/*',
    }
    r = requests.get(url, headers=headers, timeout=120, stream=True)
    print(f'  Status: {r.status_code}, Size: {len(r.content)//1024} KB')
    if r.status_code == 200 and len(r.content) > 10000:
        out_path.write_bytes(r.content)
        return True
    return False

# Re-download DR1TOT_G specifically
url = 'https://wwwn.cdc.gov/Nchs/Nhanes/2011-2012/DR1TOT_G.XPT'
out = DATA_DIR / 'DR1TOT_G.XPT'
out.unlink(missing_ok=True)  # delete the bad file first
ok = download_nhanes_v2(url, out)
print('Downloaded:', ok)

  Status: 200, Size: 20 KB
Downloaded: True


In [9]:
import pandas as pd
df = pd.read_sas(str(DATA_DIR / 'DR1TOT_G.XPT'), format='xport', encoding='utf-8')
print(f'Rows: {df.shape[0]}, Cols: {df.shape[1]}')
print(df.columns[:5].tolist())


ValueError: Header record is not an XPORT file.

In [10]:
# Check what the file actually contains
with open(DATA_DIR / 'DR1TOT_G.XPT', 'rb') as f:
    first_bytes = f.read(500)
print('File size:', (DATA_DIR / 'DR1TOT_G.XPT').stat().st_size, 'bytes')
print('First bytes:', first_bytes[:200])

File size: 20905 bytes
First bytes: b'<!DOCTYPE html>\r\n<html lang="en-us" class="cdc-2022 theme-blue cdc-root-home home-2022 cdc-page-type-2022home cdc-tp5" >\r\n<head>\r\n\t<title>Page Not Found | CDC</title>\r\n\t<meta name="description" conten'


In [11]:
import requests

# Test the correct URL format for NHANES 2011-2012
test_urls = [
    'https://wwwn.cdc.gov/Nchs/Nhanes/2011-2012/DR1TOT_G.XPT',
    'https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2011/DataFiles/DR1TOT_G.xpt',
    'https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2011/DataFiles/DR1TOT_G.XPT',
]

for url in test_urls:
    r = requests.get(url, timeout=30)
    first = r.content[:50]
    print(f'Status {r.status_code} | Size {len(r.content)//1024}KB | {url}')
    print(f'  Starts with: {first}')
    print()

Status 200 | Size 20KB | https://wwwn.cdc.gov/Nchs/Nhanes/2011-2012/DR1TOT_G.XPT
  Starts with: b'<!DOCTYPE html>\r\n<html lang="en-us" class="cdc-202'

Status 200 | Size 12133KB | https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2011/DataFiles/DR1TOT_G.xpt
  Starts with: b'HEADER RECORD*******LIBRARY HEADER RECORD!!!!!!!00'

Status 200 | Size 12133KB | https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2011/DataFiles/DR1TOT_G.XPT
  Starts with: b'HEADER RECORD*******LIBRARY HEADER RECORD!!!!!!!00'



In [12]:
import requests, time
from pathlib import Path

DATA_DIR = Path('nhanes_data')

# Correct URL pattern
# Year maps: 2011-12=2011, 2013-14=2013, 2015-16=2015, 2017-18=2017
CYCLES = {
    '2011': 'G',
    '2013': 'H',
    '2015': 'I',
    '2017': 'J',
}

FILES_PER_CYCLE = [
    'DR1TOT', 'DR2TOT', 'DEMO', 'GHB', 'TCHOL', 'HDL', 'TRIGLY', 'BMX', 'BPX'
]

def download_file(year, suffix, prefix):
    filename = f'{prefix}_{suffix}.XPT'
    url = f'https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/{year}/DataFiles/{prefix}_{suffix}.xpt'
    out_path = DATA_DIR / filename
    out_path.unlink(missing_ok=True)  # delete any bad cached version
    
    try:
        r = requests.get(url, timeout=120)
        if r.status_code == 200 and not r.content.startswith(b'<!DOCTYPE'):
            out_path.write_bytes(r.content)
            print(f'  ✓ {filename} — {len(r.content)//1024} KB')
            return True
        else:
            print(f'  ✗ {filename} — not found at this URL')
            return False
    except Exception as e:
        print(f'  ✗ {filename} — {e}')
        return False

success, failed = 0, []
for year, suffix in CYCLES.items():
    print(f'\nCycle {year}:')
    for prefix in FILES_PER_CYCLE:
        ok = download_file(year, suffix, prefix)
        if ok: success += 1
        else: failed.append(f'{prefix}_{suffix}')
        time.sleep(0.3)

print(f'\nDone: {success} downloaded, {len(failed)} failed')
if failed: print('Failed:', failed)


Cycle 2011:
  ✓ DR1TOT_G.XPT — 12133 KB
  ✓ DR2TOT_G.XPT — 6067 KB
  ✓ DEMO_G.XPT — 3665 KB
  ✓ GHB_G.XPT — 103 KB
  ✓ TCHOL_G.XPT — 184 KB
  ✓ HDL_G.XPT — 184 KB
  ✓ TRIGLY_G.XPT — 153 KB
  ✓ BMX_G.XPT — 1901 KB
  ✓ BPX_G.XPT — 1974 KB

Cycle 2013:
  ✓ DR1TOT_H.XPT — 12903 KB
  ✓ DR2TOT_H.XPT — 6528 KB
  ✓ DEMO_H.XPT — 3743 KB
  ✓ GHB_H.XPT — 110 KB
  ✓ TCHOL_H.XPT — 195 KB
  ✓ HDL_H.XPT — 195 KB
  ✓ TRIGLY_H.XPT — 157 KB
  ✓ BMX_H.XPT — 1997 KB
  ✓ BPX_H.XPT — 1767 KB

Cycle 2015:
  ✓ DR1TOT_I.XPT — 12550 KB
  ✓ DR2TOT_I.XPT — 6350 KB
  ✓ DEMO_I.XPT — 3668 KB
  ✓ GHB_I.XPT — 106 KB
  ✓ TCHOL_I.XPT — 189 KB
  ✓ HDL_I.XPT — 189 KB
  ✓ TRIGLY_I.XPT — 151 KB
  ✓ BMX_I.XPT — 1942 KB
  ✓ BPX_I.XPT — 1569 KB

Cycle 2017:
  ✓ DR1TOT_J.XPT — 11447 KB
  ✓ DR2TOT_J.XPT — 5792 KB
  ✓ DEMO_J.XPT — 3332 KB
  ✓ GHB_J.XPT — 101 KB
  ✓ TCHOL_J.XPT — 175 KB
  ✓ HDL_J.XPT — 175 KB
  ✓ TRIGLY_J.XPT — 239 KB
  ✓ BMX_J.XPT — 1431 KB
  ✓ BPX_J.XPT — 1431 KB

Done: 36 downloaded, 0 failed


In [14]:

import pandas as pd
from pathlib import Path

DATA_DIR = Path('nhanes_data')

df = pd.read_sas(str(DATA_DIR / 'DR1TOT_G.XPT'), format='xport', encoding='utf-8')
print(f'Rows: {df.shape[0]}, Cols: {df.shape[1]}')
print('Columns:', list(df.columns[:10]))

Rows: 9338, Cols: 166
Columns: ['SEQN', 'WTDRD1', 'WTDR2D', 'DR1DRSTZ', 'DR1EXMER', 'DRABF', 'DRDINT', 'DR1DBIH', 'DR1DAY', 'DR1LANG']
